# Spotify Data Cleaning: Daily Songs

* Simple pandas cleaning for `charts_songs_daily_365D.csv`. 
* The final DataFrame is `clean_songs_daily_365D`. 
* No intermediate CSV export. 
* The last cell loads into MySQL staging.
* Run cells in order. Original CSV goes in `data/raw/`.

---

## 1. Import and read source

In [11]:
import pandas as pd
from pathlib import Path

In [12]:
# Find the source file from the project folder.
def locate_raw_data(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidate = folder / "data" / "raw" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Missing CSV in data/raw/: {filename}")

songs_daily_df = pd.read_csv(locate_raw_data("charts_songs_daily_365D.csv"))
print("Source rows:", len(songs_daily_df))

Source rows: 5124674


---

## 2. Standardize source columns

In [13]:
# Keep the original dataset columns, including chart history and artist relationships.
# Rename the track URI field (daily chart date remains `date`).
songs_daily_df = songs_daily_df.rename(columns={"uri": "track_uri"})

# Dates: invalid or missing dates stay as NaT for validation and staging decisions.
for column in ["date", "entry_date", "peak_date", "release_date"]:
    if column in songs_daily_df:
        songs_daily_df[column] = pd.to_datetime(songs_daily_df[column], errors="coerce")

# Country and text standardization; do not alter proper-name capitalization.
for column in ["country", "track_name", "label", "artist_names", "entry_status"]:
    if column in songs_daily_df:
        songs_daily_df[column] = songs_daily_df[column].astype("string").str.strip()
        
# Normalize country abbreviations to uppercase.
if "country" in songs_daily_df:
    songs_daily_df["country"] = songs_daily_df["country"].str.upper()
if "entry_status" in songs_daily_df:
    songs_daily_df["entry_status"] = songs_daily_df["entry_status"].str.replace("_", " ", regex=False)

# Fix the original 'spotify:tack:' typo; remove only a prefix at the start.
songs_daily_df["track_uri"] = (songs_daily_df["track_uri"].astype("string")
                              .str.strip().str.replace(r"^spotify:track:", "", regex=True))

# Preserve artist_uris because it may contain multiple artist IDs for bridge_track_artists.
if "artist_uris" in songs_daily_df:
    songs_daily_df["artist_uris"] = (songs_daily_df["artist_uris"].astype("string")
        .str.strip().str.replace("spotify:artist:", "", regex=False))

# Unknown streaming volume stays null; never invent zero streams.
# Convert streams to a nullable integer type.
if "streams" in songs_daily_df:
    songs_daily_df["streams"] = pd.to_numeric(songs_daily_df["streams"], errors="coerce").astype("Int64")


---

## 3. Handle missing values and duplicates

In [14]:
# One observation per chart key; artist master uses one row per artist URI.
keys = ["date", "country", "track_uri"]
required = ["date", "country", "track_uri", "rank"]
expected_columns = ['date', 'country', 'rank', 'track_uri', 'artist_names', 'track_name', 'label', 'peak_rank', 'previous_rank', 'days_on_chart', 'streams', 'consecutive_days', 'entry_status', 'peak_date', 'entry_rank', 'entry_date', 'release_date', 'artist_uris']
missing_columns = set(expected_columns) - set(songs_daily_df.columns)
if missing_columns:
    raise ValueError(f"Missing staging columns: {sorted(missing_columns)}")

# Standardize blank text. Keep optional unknown measures as SQL NULL.
for col in songs_daily_df.select_dtypes(include=["object", "string"]).columns:
    songs_daily_df[col] = songs_daily_df[col].astype("string").str.strip().replace("", pd.NA)

# Descriptive labels have meaningful defaults; unknown statistics do not.
for col, default in {'track_name': 'Unknown Track', 'artist_names': 'Unknown Artist', 'label': 'Unknown Label'}.items():
    songs_daily_df[col] = songs_daily_df[col].fillna(default)

# Invalid optional ranks/counts are unknown, not zero.
for col in ['peak_rank', 'entry_rank', 'previous_rank']:
    values = pd.to_numeric(songs_daily_df[col], errors="coerce")
    songs_daily_df[col] = values.where(values > 0).astype("Int64")
for col in ['streams', 'days_on_chart', 'consecutive_days']:
    values = pd.to_numeric(songs_daily_df[col], errors="coerce")
    songs_daily_df[col] = values.where(values >= 0).astype("Int64")

# Required chart keys cannot be invented; keep only valid observations.
valid = songs_daily_df[required].notna().all(axis=1) & songs_daily_df["rank"].between(1, 200).fillna(False)

# Keep optional chart attributes within the staging constraints.
for col in ["peak_rank", "entry_rank"]:
    songs_daily_df[col] = songs_daily_df[col].where(songs_daily_df[col].between(1, 200))
songs_daily_df["consecutive_days"] = songs_daily_df["consecutive_days"].where(
    songs_daily_df["days_on_chart"].isna() | (songs_daily_df["consecutive_days"] <= songs_daily_df["days_on_chart"])
)

invalid_count = int((~valid).sum())
songs_daily_df = songs_daily_df.loc[valid]

# Fast single-pass grain deduplication. Keep first original source row.
duplicate_count = int(songs_daily_df.duplicated(subset=keys).sum())
clean_songs_daily_365D = songs_daily_df.drop_duplicates(subset=keys, keep="first").reset_index(drop=True)

print("Invalid key/rank rows removed:", invalid_count)
print("Repeated business keys removed:", duplicate_count)
print("Final staging rows:", len(clean_songs_daily_365D))


Invalid key/rank rows removed: 0
Repeated business keys removed: 20
Final staging rows: 5124654


---

## 4. Review and load to MySQL

In [15]:
print("DataFrame:", "clean_songs_daily_365D")
print("Rows:", len(clean_songs_daily_365D))
print("Missing values (optional columns):")
print(clean_songs_daily_365D.isna().sum().to_string())
display(clean_songs_daily_365D.head(5))

DataFrame: clean_songs_daily_365D
Rows: 5124654
Missing values (optional columns):
date                     0
country                  0
rank                     0
track_uri                0
artist_names             0
track_name               0
label                    0
peak_rank                0
previous_rank       409880
days_on_chart            0
streams              18600
consecutive_days         0
entry_status             0
peak_date                0
entry_rank               0
entry_date               0
release_date        120617
artist_uris              0


,date,country,rank,track_uri,artist_names,track_name,label,peak_rank,previous_rank,days_on_chart,streams,consecutive_days,entry_status,peak_date,entry_rank,entry_date,release_date,artist_uris
0,2025-11-11,AE,1,27xkOIER6uDLKALIelHylZ,Jin,Don’t Say You Love Me,BIGHIT MUSIC,1,18,180,31049,180,MOVED UP,2025-05-20,2,2025-05-16,2025-05-16,5vV3bFXnN6D6N3Nj4xRvaV
1,2025-11-11,AE,2,53iuhJlwXhSER5J2IYYv1W,Taylor Swift,The Fate of Ophelia,Taylor Swift,1,1,40,18896,40,MOVED DOWN,2025-10-03,1,2025-10-03,2025-10-03,06HL4z0CvFAxyc27GXpf02
2,2025-11-11,AE,3,1qbmS6ep2hbBRaEZFpn7BX,Olivia Dean,Man I Need,PLG - Capitol,2,2,83,16082,83,MOVED DOWN,2025-11-07,180,2025-08-21,2025-08-15,00x1fYSGhdqScXBRpSj3DW
3,2025-11-11,AE,4,1CPZ5BxNNd0n0nF4Orb9JS,HUNTR/X|EJAE|AUDREY NUNA|REI AMI|KPop Demon Hu...,Golden,K-Pop Demon Hunters,1,3,142,13429,142,MOVED DOWN,2025-08-12,91,2025-06-23,2025-06-20,2yNNYQBChuox9A5Ka93BIn|0RMJOzHDhAKY1o2j0W0vxY|...
4,2025-11-11,AE,5,3yWuTOYDztXjZxdE2cIRUa,Taylor Swift,Opalite,Taylor Swift,1,4,40,12554,40,MOVED DOWN,2025-10-22,3,2025-10-03,2025-10-03,06HL4z0CvFAxyc27GXpf02


In [16]:
# Import the shared staging loader from etl/load.py.
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "etl" / "load.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Missing etl/load.py in the project folder")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from etl.load import load_dataframe

In [17]:
# Load only the cleaned DataFrame; no exclusion-triggered ValueError.
if clean_songs_daily_365D.empty:
    raise ValueError("No valid records available for staging.")
load_dataframe(clean_songs_daily_365D, "clean_songs_daily_365D")
print("Loaded:", "clean_songs_daily_365D", "| Rows:", len(clean_songs_daily_365D))

Loaded 5,124,654 rows into spotify_analytics_stg.clean_songs_daily_365D
Loaded: clean_songs_daily_365D | Rows: 5124654
